In [1]:
%%time
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from math import nan
from pathlib import Path
import dask
import yaml
import os
import warnings
import gc
import textwrap
import glob

from dask_jobqueue import PBSCluster
from dask.distributed import Client

warnings.filterwarnings('ignore')

CPU times: user 702 ms, sys: 152 ms, total: 855 ms
Wall time: 2.23 s


In [2]:
%%time
# Configure Dask to use a PBSCluster on Casper
dask.config.set({"distributed.scheduler.worker_saturation": 1.0})

cluster = PBSCluster(
    cores=1,
    memory='30GB',
    processes=1,
    queue='casper',
    local_directory='$TMPDIR',
    resource_spec='select=1:ncpus=1:mem=30GB',
    project='P04010022',
    walltime='24:00:00',
    interface='mgt',
    job_extra_directives=[
        "-o dask_logs/dask-worker.$PBS_JOBID.out",
        "-e dask_logs/dask-worker.$PBS_JOBID.err",
    ]
)

# Adjust number of workers as needed (start modestly to avoid hammering the filesystem)
cluster.scale(16)

# Dask dashboard URL for JupyterHub
dask.config.set({
    'distributed.dashboard.link':
    'https://jupyterhub.hpc.ucar.edu/stable/user/{USER}/proxy/{port}/status'
})

client = Client(cluster)
client

CPU times: user 491 ms, sys: 77.4 ms, total: 568 ms
Wall time: 15.9 s


Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/hannay/proxy/42135/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/hannay/proxy/42135/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://10.18.206.68:35365,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/hannay/proxy/42135/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [4]:
cluster

PBSCluster(1c0c76b1, 'tcp://10.18.206.68:35365', workers=2, threads=2, memory=55.88 GiB)

In [5]:
%%time
# CAM HR input files
h5a = sorted(glob.glob(
    "/glade/campaign/cesm/community/amwg/cam_hr/"
    "f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001/atm/hist/*.h5a.*199*"
))

h5i = sorted(glob.glob(
    "/glade/campaign/cesm/community/amwg/cam_hr/"
    "f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001/atm/hist/*.h5i.*199*"
))

print("Number of h5a files:", len(h5a))
print("Number of h5i files:", len(h5i))

if len(h5a) != len(h5i):
    raise ValueError("h5a and h5i file lists have different lengths!")

Number of h5a files: 244
Number of h5i files: 244
CPU times: user 6.55 ms, sys: 3.57 ms, total: 10.1 ms
Wall time: 11.6 ms


In [6]:
%%time
import os, gc
import numpy as np
import xarray as xr

# Constants for Tb calculation
sigma = 5.67e-8        # W m-2 K-4
a = 1.228
b = -1.106e-3          # K-1

h5avars = ['PRECT']

outdir = "/glade/derecho/scratch/hannay/MOAPP/hourly_files/"
os.makedirs(outdir, exist_ok=True)

def process_one_pair(ifile):
    """Process a single (h5a, h5i) pair into CAM7_L58_HR_hourly.YYYY-MM-DD.nc"""
    print("=====")

    fname = os.path.basename(h5i[ifile])

    # Extract timestamp: '2000-01-11' from '2000-01-11-03600'
    timestamp = fname.split(".")[-2]       # '2000-01-11-03600'
    date_str = timestamp.rsplit("-", 1)[0] # '2000-01-11'

    # Build output filename
    new_fname = f"CAM7_L58_HR_hourly.{date_str}.nc"
    outpath = os.path.join(outdir, new_fname)

    # Skip if file already exists
    if os.path.exists(outpath):
        print(f"File already exists, skipping: {outpath}")
        return outpath

    print(f"Creating: {outpath}")

    # --- Open h5a and convert PRECT ---
    h5adat = xr.open_dataset(h5a[ifile], chunks={"time": 24}, engine="netcdf4")
    h5adat = h5adat[h5avars]

    # m/s → mm/hr
    h5adat['PRECT'] = h5adat['PRECT'] * 1000.0 * 3600.0
    h5adat['PRECT'] = h5adat['PRECT'].astype('float32')

    # Set PRECT attributes exactly as you want
    h5adat['PRECT'].attrs = {
        "long_name": "Total (convective and large-scale) precipitation rate (liq + ice)",
        "units": "mm hr-1",
        "cell_methods": "time: mean",
    }
    # Set _FillValue = NaNf
    h5adat['PRECT'].encoding['_FillValue'] = np.float32(np.nan)

    # --- Open h5i and compute Tb ---
    h5idat = xr.open_dataset(h5i[ifile], chunks={"time": 24}, engine="netcdf4")

    FLUT = h5idat['FLUT']

    Tf = (FLUT / sigma) ** 0.25
    Tb = (-a + np.sqrt(a**2 + 4.0 * b * Tf)) / (2.0 * b)

    Tb = Tb.rename('Tb')
    Tb.attrs['long_name'] = "flux equivalent brightness temperature"
    Tb.attrs['units'] = "K"
    h5idat['Tb'] = Tb

    # Use the time axis from h5i for the merged data
    h5adat['time'] = h5idat.time.values

    # Merge datasets
    newdat = xr.merge([h5adat, h5idat])

    # --- Rename spatial dimensions lat -> y, lon -> x (if present) ---
    rename_dict = {}
    if 'lat' in newdat.dims:
        rename_dict['lat'] = 'y'
    if 'lon' in newdat.dims:
        rename_dict['lon'] = 'x'
    if rename_dict:
        newdat = newdat.rename(rename_dict)

    # --- Build 2D X(y,x) and Y(y,x) from 1D coords ---

    # Latitude coordinate (1D)
    if 'lat' in newdat:
        lat_1d = newdat['lat']
    elif 'y' in newdat:
        lat_1d = newdat['y']
    else:
        raise KeyError("No latitude coordinate found (expected 'lat' or 'y').")

    # Longitude coordinate (1D)
    if 'lon' in newdat:
        lon_1d = newdat['lon']
    elif 'x' in newdat:
        lon_1d = newdat['x']
    else:
        raise KeyError("No longitude coordinate found (expected 'lon' or 'x').")

    # Broadcast to 2D (y, x): Y(y,x) from lat, X(y,x) from lon
    Y_2d, X_2d = xr.broadcast(lat_1d, lon_1d)
    Y_2d = Y_2d.rename('Y')
    X_2d = X_2d.rename('X')

    Y_2d.attrs['long_name'] = 'latitude on (y,x) grid'
    Y_2d.attrs['units'] = lat_1d.attrs.get('units', 'degrees_north')

    X_2d.attrs['long_name'] = 'longitude on (y,x) grid'
    X_2d.attrs['units'] = lon_1d.attrs.get('units', 'degrees_east')

    newdat['Y'] = Y_2d
    newdat['X'] = X_2d

    # --- Create 1D lat(y) and lon(x) variables with desired attrs ---

    # Get 1D values along y/x dims
    if 'y' in newdat.coords:
        y_vals = newdat['y'].values
    else:
        # if rename didn't change the coord name, fall back
        y_vals = lat_1d.values

    if 'x' in newdat.coords:
        x_vals = newdat['x'].values
    else:
        x_vals = lon_1d.values

    # Drop y/x coordinate variables but keep dimensions 'y' and 'x'
    drop_vars = [v for v in ['y', 'x'] if v in newdat.coords]
    if drop_vars:
        newdat = newdat.drop_vars(drop_vars)

    # Create lat(y) and lon(x)
    lat_da = xr.DataArray(
        y_vals,
        dims=('y',),
        name='lat',
        attrs={
            'long_name': 'latitude',
            'units': 'degrees_north',
        },
    )
    lon_da = xr.DataArray(
        x_vals,
        dims=('x',),
        name='lon',
        attrs={
            'long_name': 'longitude',
            'units': 'degrees_east',
        },
    )

    # Set FillValue for lat/lon (NaN as double)
    lat_da.encoding['_FillValue'] = np.nan
    lon_da.encoding['_FillValue'] = np.nan

    # Attach them as coordinates
    newdat = newdat.assign_coords(lat=lat_da, lon=lon_da)

    # --- Write out ---
    newdat.to_netcdf(outpath, format='NETCDF4')

    newdat.close()
    h5adat.close()
    h5idat.close()

    # Cleanup
    del newdat, h5adat, h5idat, FLUT, Tf, Tb, Y_2d, X_2d
    gc.collect()

    return outpath


CPU times: user 829 μs, sys: 189 μs, total: 1.02 ms
Wall time: 2.98 ms


In [7]:
%%time
# Optional: test a single file before running in parallel
_ = process_one_pair(0)

=====
Creating: /glade/derecho/scratch/hannay/MOAPP/hourly_files/CAM7_L58_HR_hourly.1990-01-01.nc
CPU times: user 5.86 s, sys: 1.05 s, total: 6.92 s
Wall time: 1min 58s


In [8]:
%%time
from dask import delayed
from dask.distributed import wait

nfiles = len(h5a)
print("Total files:", nfiles)

# For testing, you can restrict indices, e.g. range(10)
indices = range(nfiles)

tasks = [delayed(process_one_pair)(i) for i in indices]

futures = client.compute(tasks)
wait(futures)

print("All requested files processed.")

Total files: 244
All requested files processed.
CPU times: user 6min 44s, sys: 52.9 s, total: 7min 37s
Wall time: 1h 48min 35s


In [7]:
%%time
from dask import delayed
from dask.distributed import wait

nfiles = len(h5a)
print("Total files:", nfiles)

# For testing, you can restrict indices, e.g. range(10)
indices = range(nfiles)

tasks = [delayed(process_one_pair)(i) for i in indices]

futures = client.compute(tasks)
wait(futures)

print("All requested files processed.")

Total files: 25
All requested files processed.
CPU times: user 1min 33s, sys: 11.8 s, total: 1min 45s
Wall time: 26min 10s


In [9]:
cluster.close()
client.close()
